# Lunar Thermal Model — Animations

Companion to `Lunar_Thermal_Presentation.ipynb`.

Requires that notebook to have been run first so that `APOLLO_DATA`, `Z_GRID`,
`APOLLO_RESULTS`, and `COMPARE_RESULTS` are available in the kernel session,
**or** run the setup cell below to load them independently.

All GIFs are saved to the `gifs/` folder and displayed inline.
If a GIF already exists it is displayed immediately without re-generating.

In [ ]:
# ── Setup (run this if not continuing from the main notebook) ────────────────
import shutil, os
os.chdir(os.path.dirname(os.path.abspath(r'/Users/rp3gregorio/Desktop/Research/Lunar-Clean/Lunar_Thermal_Animations.ipynb')))

import numpy as np
import matplotlib.pyplot as plt
import time

import lunar.dem        as dem
import lunar.horizon    as horizon
import lunar.solver     as solver
import lunar.models     as models
import lunar.analysis   as analysis
import lunar.plots      as plots

from lunar.constants import LUNAR_DAY

# ── Re-run configuration if needed ───────────────────────────────────────────
# If you ran Lunar_Thermal_Presentation.ipynb first in the SAME kernel session,
# all variables below already exist and this cell is a no-op.
if 'APOLLO_DATA' not in dir():
    exec(open('_anim_setup.py').read()) if os.path.exists('_anim_setup.py') else print(
        'Run Lunar_Thermal_Presentation.ipynb first (Cells 1-4), then re-run this cell.')
else:
    print('Main notebook variables detected — skipping re-run.')

---
## Animation 1 — Temperature Depth Profile T(z)

Shows the full 1-D temperature profile evolving through one lunar day.
Surface heats up at noon and cools at night; deeper layers respond with
a time delay (thermal phase lag).

In [ ]:
from IPython.display import Image, display
import matplotlib.animation as animation

gif_path1 = os.path.join('gifs', 'thermal_profile_animation.gif')

if os.path.exists(gif_path1) and os.path.getsize(gif_path1) > 5000:
    display(Image(filename=gif_path1))
else:
    print('Generating T(z) animation ...')
    fig_anim, ax_anim = plt.subplots(figsize=(7, 6))
    _cyc_a15_anim = APOLLO_DATA['Apollo 15']['disc']['cycles']
    _tp_a15 = APOLLO_DATA['Apollo 15']['disc']['T_profile']
    _ta_a15 = APOLLO_DATA['Apollo 15']['disc']['T_arr']
    from lunar.constants import LUNAR_DAY
    _t_start = _ta_a15[-1] - LUNAR_DAY
    _idx     = np.where(_ta_a15 >= _t_start)[0]
    _t_h     = (_ta_a15[_idx] - _t_start) / 3600.0
    _frames  = np.linspace(0, len(_idx)-1, 60, dtype=int)
    def _update(fi):
        ax_anim.cla()
        ti = _idx[_frames[fi]]
        ax_anim.plot(_tp_a15[ti, :], Z_GRID * 100, 'b-', lw=2)
        ax_anim.set_xlabel('Temperature (K)'); ax_anim.set_ylabel('Depth (cm)')
        ax_anim.set_title(f'T(z) profile  LLT = {_t_h[_frames[fi]]:.0f} h')
        ax_anim.invert_yaxis()
    ani = animation.FuncAnimation(fig_anim, _update, frames=60, interval=100)
    ani.save(gif_path1, writer='pillow', fps=10)
    plt.close(fig_anim)
    display(Image(filename=gif_path1))
    print(f'Saved: {gif_path1}')

---
## Animation 2 — Heatmap + Profile Side-by-Side

Left panel: growing T(z, t) heatmap (depth × LLT).
Right panel: instantaneous T(z) profile at the current frame time.
Together they show how the thermal wave builds up through the day.

In [ ]:
from IPython.display import Image, display
import matplotlib.animation as animation
import matplotlib.gridspec as gridspec

gif_path2 = os.path.join('gifs', 'heatmap_animation.gif')

if os.path.exists(gif_path2) and os.path.getsize(gif_path2) > 5000:
    display(Image(filename=gif_path2))
else:
    print('Generating heatmap animation ...')
    _tp = APOLLO_DATA['Apollo 15']['disc']['T_profile']
    _ta = APOLLO_DATA['Apollo 15']['disc']['T_arr']
    from lunar.constants import LUNAR_DAY
    _t_start = _ta[-1] - LUNAR_DAY
    _idx     = np.where(_ta >= _t_start)[0]
    _t_h     = (_ta[_idx] - _t_start) / 3600.0
    _frames  = np.linspace(0, len(_idx)-1, 60, dtype=int)
    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 6),
                               gridspec_kw={'width_ratios': [2, 1]})
    def _update2(fi):
        for ax in axes2: ax.cla()
        ti   = _idx[_frames[fi]]
        llt  = _t_h[_frames[fi]]
        T2d  = _tp[_idx[:_frames[fi]+1], :]
        axes2[0].contourf((_ta[_idx[:_frames[fi]+1]] - _t_start)/3600,
                          Z_GRID*100, T2d.T, levels=30, cmap='hot')
        axes2[0].set_xlabel('LLT (h)'); axes2[0].set_ylabel('Depth (cm)')
        axes2[0].invert_yaxis(); axes2[0].set_title('T(z,t) heatmap')
        axes2[1].plot(_tp[ti,:], Z_GRID*100, 'b-', lw=2)
        axes2[1].set_xlabel('T (K)')
        axes2[1].invert_yaxis(); axes2[1].set_title(f'Profile at {llt:.0f} h')
    ani2 = animation.FuncAnimation(fig2, _update2, frames=60, interval=120)
    ani2.save(gif_path2, writer='pillow', fps=8)
    plt.close(fig2)
    display(Image(filename=gif_path2))
    print(f'Saved: {gif_path2}')

---
## Animation 3 — Diurnal Anomaly Wave Propagating with Depth

Plots T(z) − T_mean(z): the temperature anomaly wave.
Blue = nightside frame, red = dayside frame.
Watch how the wave shrinks in amplitude and shifts in phase as it
travels deeper into the regolith.

In [ ]:
# ── Animated GIF: diurnal temperature anomaly wave sweeping through depth ──────
# Each frame = one LLT time step. Shows:
#   - T anomaly vs depth profile — the wave propagating downward
#   - Coloured by dayside/nightside

from IPython.display import Image, display
import matplotlib.animation as animation
from lunar.lunar_lst import load_kernels as _load_spice
from lunar.constants import LUNAR_DAY

_load_spice()

gif_path3 = os.path.join('gifs', 'diurnal_wave_Apollo_15.gif')
gif_path4 = os.path.join('gifs', 'diurnal_wave_Apollo_17.gif')

for site_name, gif_path in [('Apollo 15', gif_path3), ('Apollo 17', gif_path4)]:
    if os.path.exists(gif_path) and os.path.getsize(gif_path) > 5000:
        print(f'{site_name}: displaying existing GIF')
        display(Image(filename=gif_path))
        continue
    print(f'Generating diurnal wave animation for {site_name} ...')
    _sd  = APOLLO_DATA[site_name]
    _tp  = _sd['disc']['T_profile']
    _ta  = _sd['disc']['T_arr']
    _t0  = _ta[-1] - LUNAR_DAY
    _idx = np.where(_ta >= _t0)[0]
    _t_h = (_ta[_idx] - _t0) / 3600.0
    _nf  = 72
    _fi  = np.linspace(0, len(_idx)-1, _nf, dtype=int)
    day_h = LUNAR_DAY / 3600.0
    fig_w, ax_w = plt.subplots(figsize=(6, 7))
    def _upd(k):
        ax_w.cla()
        ti  = _idx[_fi[k]]
        llt = _t_h[_fi[k]]
        T   = _tp[ti, :]
        Tm  = _tp[_idx, :].mean(axis=0)
        ax_w.plot(T - Tm, Z_GRID * 100, lw=2.5,
                  color='#C0392B' if llt < day_h*0.75 and llt > day_h*0.25 else '#2980B9')
        ax_w.axvline(0, color='#888', lw=0.8, ls=':')
        ax_w.set_xlabel('T anomaly (K)', fontsize=11)
        ax_w.set_ylabel('Depth (cm)', fontsize=11)
        ax_w.invert_yaxis()
        phase = 'Day' if day_h*0.25 < llt < day_h*0.75 else 'Night'
        ax_w.set_title(f'{site_name}  |  LLT = {llt:.0f} h  [{phase}]', fontsize=12)
        ax_w.set_xlim(-35, 35)
    ani_w = animation.FuncAnimation(fig_w, _upd, frames=_nf, interval=120)
    ani_w.save(gif_path, writer='pillow', fps=8)
    plt.close(fig_w)
    display(Image(filename=gif_path))
    print(f'Saved: {gif_path}')

---
## Animation 4 — Diurnal Temperature Curves by Depth (Discrete vs Hayne)

Two side-by-side animated GIFs sweep through the lunar day one time-step at a
time. Each frame adds the current LLT position as a moving vertical line over
the full diurnal temperature curves at every sensor depth.
Left panel = Discrete model, Right panel = Hayne 2017 model.
Both use the Apollo 15 site geometry computed in the main notebook.


In [ ]:
# ── Animation 4: Diurnal T curves by depth, sweeping LLT marker ─────────────
# SELF-CONTAINED: runs its own lightweight simulation (flat horizon, no DEM).
# Uses the same physical parameters as the main notebook (Cell 2).
# Apollo 15 site  |  Left = Discrete model  |  Right = Hayne 2017 model

from IPython.display import Image, display
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import os

import lunar.solver   as solver
import lunar.models   as models
import lunar.analysis as analysis
from lunar.constants import LUNAR_DAY

gif_path4 = os.path.join('gifs', 'diurnal_by_depth_a15.gif')
os.makedirs('gifs', exist_ok=True)

if os.path.exists(gif_path4):
    print(f'GIF already exists: {gif_path4}')
    display(Image(filename=gif_path4))
else:
    # ── Simulation parameters (mirror Cell 2 of main notebook) ────────────────
    LAT_A15   = 26.1323
    LON_A15   =  3.6285
    T_SURF    = 250.0       # geothermal init temperature (K)
    CHI       = 2.7
    SUNSCALE  = 1.10
    ALBEDO    = 0.09
    EMISSIVITY= 0.95
    NDAYS     = 20
    H_PARAM   = 0.07
    DISC_ID   = models.MODEL_ID_MAP['discrete']
    HAYNE_ID  = models.MODEL_ID_MAP['hayne_exponential']
    models.set_hayne_h(H_PARAM)
    models.set_layer1_h(H_PARAM)

    # Sensor depths matching Apollo 15 probe depths (metres)
    SENSOR_DEPTHS = [0.0, 0.05, 0.35, 0.49, 0.84, 0.91, 1.01, 1.29, 1.39]

    # Flat horizon (no DEM needed — keeps this cell self-contained)
    Z_GRID  = solver.create_depth_grid()
    N_AZ    = 72
    AZ_ANGS = np.linspace(0, 2*np.pi, N_AZ, endpoint=False, dtype=np.float32)
    HORIZ   = np.zeros(N_AZ, dtype=np.float32)   # flat = no obstruction
    SLOPE   = 0.0
    ASPECT  = 0.0

    print('Running Discrete model for Apollo 15 ...', end=' ', flush=True)
    T_init_d = solver.compute_equilibrium_profile(Z_GRID, T_SURF, DISC_ID, CHI)
    TP_d, TA_d = solver.solve_thermal_model(
        Z_GRID, T_init_d, LAT_A15, LON_A15, SLOPE, ASPECT,
        HORIZ, AZ_ANGS, CHI, DISC_ID, SUNSCALE, NDAYS,
        albedo=ALBEDO, emissivity=EMISSIVITY)
    cyc_disc = analysis.get_diurnal_cycles(TP_d, TA_d, Z_GRID, depths_m=SENSOR_DEPTHS)
    print('done')

    print('Running Hayne 2017 model for Apollo 15 ...', end=' ', flush=True)
    T_init_h = solver.compute_equilibrium_profile(Z_GRID, T_SURF, HAYNE_ID, CHI)
    TP_h, TA_h = solver.solve_thermal_model(
        Z_GRID, T_init_h, LAT_A15, LON_A15, SLOPE, ASPECT,
        HORIZ, AZ_ANGS, CHI, HAYNE_ID, SUNSCALE, NDAYS,
        albedo=ALBEDO, emissivity=EMISSIVITY)
    cyc_hayne = analysis.get_diurnal_cycles(TP_h, TA_h, Z_GRID, depths_m=SENSOR_DEPTHS)
    print('done')

    # ── Build curves ───────────────────────────────────────────────────────────
    day_h = LUNAR_DAY / 3600.0
    depth_colors = [
        '#D73027', '#FC8D59', '#FEE090',
        '#ABD9E9', '#74ADD1', '#4575B4', '#313695', '#1A1A6C',
    ]

    def _build_curves(cyc):
        out = []
        for d, data in sorted(cyc.items()):
            t = np.asarray(data['time_h'], float)
            T = np.asarray(data['temperature'], float)
            s = np.argsort(t)
            out.append((d, t[s], T[s], f'{d*100:.1f} cm'))
        return out

    curves_d = _build_curves(cyc_disc)
    curves_h = _build_curves(cyc_hayne)

    # ── Build figure ───────────────────────────────────────────────────────────
    fig, (ax_d, ax_h) = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

    def _setup_ax(ax, curves, title):
        for i, (d, t, T, lbl) in enumerate(curves):
            ax.plot(t, T, lw=1.8,
                    color=depth_colors[i % len(depth_colors)],
                    label=lbl, zorder=3)
        half = day_h / 2
        ax.axvspan(0,         half*0.48, color='#1a1a2e', alpha=0.10, zorder=0)
        ax.axvspan(half*1.52, day_h,     color='#1a1a2e', alpha=0.10, zorder=0)
        ax.axvline(half, color='#888', lw=0.8, ls='--', alpha=0.6)
        ax.text(half + day_h*0.01, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 400,
                'Noon', fontsize=8, color='#666', va='top')
        ax.set_xlim(0, day_h)
        _ticks = [0, day_h/4, day_h/2, 3*day_h/4, day_h]
        _lbls  = ['Midnight\n(0h)',
                  'Sunrise\n({:.0f}h)'.format(day_h/4),
                  'Noon\n({:.0f}h)'.format(day_h/2),
                  'Sunset\n({:.0f}h)'.format(3*day_h/4),
                  'Midnight\n({:.0f}h)'.format(day_h)]
        ax.set_xticks(_ticks)
        ax.set_xticklabels(_lbls, fontsize=8)
        ax.set_xlabel('Local Lunar Time (hours)', fontsize=10)
        ax.set_ylabel('Temperature (K)', fontsize=10)
        ax.set_title(title, fontsize=11, weight='bold')
        ax.legend(loc='upper right', fontsize=7, title='Depth',
                  framealpha=0.85, title_fontsize=8)
        ax.grid(True, alpha=0.20)

    _setup_ax(ax_d, curves_d, 'Discrete Model — Apollo 15 (flat horizon)')
    _setup_ax(ax_h, curves_h, 'Hayne 2017 Model — Apollo 15 (flat horizon)')
    fig.suptitle('Apollo 15 — Diurnal Temperature Cycles by Depth',
                 fontsize=13, weight='bold', y=1.01)
    plt.tight_layout()

    # ── Animated time marker ───────────────────────────────────────────────────
    N_FRAMES = 120
    t_frames = np.linspace(0, day_h, N_FRAMES, endpoint=False)

    ylim_d = ax_d.get_ylim()
    ylim_h = ax_h.get_ylim()
    vline_d = ax_d.axvline(0, color='#E74C3C', lw=2.0, ls='-', zorder=6)
    vline_h = ax_h.axvline(0, color='#E74C3C', lw=2.0, ls='-', zorder=6)
    txt_d   = ax_d.text(0, ylim_d[1], '', ha='center', va='bottom',
                        fontsize=8, color='#E74C3C', fontweight='bold')
    txt_h   = ax_h.text(0, ylim_h[1], '', ha='center', va='bottom',
                        fontsize=8, color='#E74C3C', fontweight='bold')

    def _phase_label(t):
        frac = t / day_h
        if frac < 0.02 or frac > 0.98: return 'Midnight'
        if abs(frac - 0.25) < 0.02:    return 'Sunrise'
        if abs(frac - 0.50) < 0.02:    return 'Noon'
        if abs(frac - 0.75) < 0.02:    return 'Sunset'
        return f'{t:.0f}h'

    def _update(frame):
        t = t_frames[frame]
        vline_d.set_xdata([t, t])
        vline_h.set_xdata([t, t])
        lbl = _phase_label(t)
        txt_d.set_position((t, ylim_d[1]))
        txt_d.set_text(lbl)
        txt_h.set_position((t, ylim_h[1]))
        txt_h.set_text(lbl)
        return vline_d, vline_h, txt_d, txt_h

    ani = animation.FuncAnimation(
        fig, _update, frames=N_FRAMES, interval=80, blit=True)
    ani.save(gif_path4, writer='pillow', fps=15, dpi=120)
    plt.close(fig)
    print(f'Saved: {gif_path4}')
    display(Image(filename=gif_path4))
